In [1]:
import numpy as np
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.core.metric import Metric
from ax.core.outcome_constraint import OutcomeConstraint
from ax.core.types import ComparisonOp
from ax.modelbridge.generation_strategy import GenerationStrategy, GenerationStep
from ax.modelbridge.registry import Models
import matplotlib.pyplot as plt

In [2]:
def x_normalizer(X, var_array):
    def max_min_scaler(x, x_max, x_min):
        return (x-x_min)/(x_max-x_min)
    x_norm = []
    for x in (X):
           x_norm.append([max_min_scaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
    return x_norm

def x_denormalizer(x_norm, var_array):
    def max_min_rescaler(x, x_max, x_min):
        return x*(x_max-x_min)+x_min
    x_original = []
    for x in (x_norm):
           x_original.append([max_min_rescaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
    return x_original

def get_closest_value(given_value, array_list):
    absolute_difference_function = lambda list_value : abs(list_value - given_value)
    closest_value = min(array_list, key=absolute_difference_function)
    return closest_value

def get_closest_array(suggested_x, var_list):
    modified_array = []
    for x in suggested_x:
        modified_array.append([get_closest_value(x[i], var_list[i]) for i in range(len(x))])
    return np.array(modified_array)

In [15]:
import ollama

def query_llm(prompt, context="", model="llama3.2"):
    full_prompt = f"{context}\n\n{prompt}" if context else prompt
        
    response = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': full_prompt}],
        options={
            'temperature': 0.1,  # Low temperature for consistency
            'top_p': 1.0,       # Full vocabulary consideration
            'seed': 42          # Fixed seed for reproducibility
        }
    )
    return response['message']['content']

def load_data_context(excel_path):
    data = pd.read_excel(excel_path)
    data_context = f"""
DETAILED EXPERIMENTAL RESULTS:
"""
    for idx, row in data.iterrows():
        data_context += f"Exp {idx}, parameter: {row['Spin Speed [rpm]']:.0f}rpm, {row['Precursor Concentration [mol/L]']:.2f}M, {row['Annealing Temperature [degC]']:.0f}°C, results: Coverage={row['Film coverage']:.3f}, Uniformity={row['Uniformity']:.1f}, Phase purity={row['Phase purity']:.0f}\n"

    data_context += f"""
Coverage is measured as a fraction from 0 to 1 (higher is better).
Uniformity is measured as an average variance of film pixel spectra at three hundred wavelengths (lower is better).
Phase purity is a integer variable where 1 indicates single phase (desired) and numbers larger than 1 indicates mixed phase (undesired).
"""
    return data_context


def extract_bo_knowledge(data_context):
    
    prompt = """
The synthesis bayesian optimization aims to achieve high film coverage (maximize) and high uniformity (minimize) with single phase (phase purity = 1). Based on the experimental results, analyze how parameters affect coverage, uniformity and phase purity, respectively. Propose constraints for each parameter to narrow down the search space.

The original parameter space is:
Spin speed(rpm): 500 - 5000, step 500
Precursor concentration(M): 0.1 - 1.5, step 0.1
Annealing temperature(°C): 100 - 300, step 10

Provide a concise summary following this EXACT format:

- Coverage:
   spin speed:
   precursor concentration:
   annealing temperature:
- Uniformity:
    spin speed:
    precursor concentration:
    annealing temperature:
- Phase purity:
    spin speed:
    precursor concentration:
    annealing temperature:
If there is no clear trend, write "No clear trend".

Proposed constraints for each parameter:
- Spin speed(rpm):
- Reasoning:
- Precursor concentration(M):
- Reasoning:
- Annealing temperature(°C):
- Reasoning: 

Be specific with numerical ranges (e.g., "≤0.3 M", "≥3000 rpm").
"""
    exp_knowledge = query_llm(prompt, data_context)
    return exp_knowledge

In [16]:
data_context = load_data_context(excel_path = "/Users/shengfang/Desktop/TRI/test_FAPbI3/LLM_BO_initialization_round_summary.xlsx")
print(data_context)
exp_knowledge = extract_bo_knowledge(data_context)
print("\nExtracted Experimental Knowledge:\n", exp_knowledge)


DETAILED EXPERIMENTAL RESULTS:
Exp 0, parameter: 2000rpm, 0.60M, 250°C, results: Coverage=0.936, Uniformity=7.1, Phase purity=2
Exp 1, parameter: 500rpm, 1.30M, 150°C, results: Coverage=0.783, Uniformity=5.5, Phase purity=2
Exp 2, parameter: 2000rpm, 1.40M, 270°C, results: Coverage=0.984, Uniformity=5.0, Phase purity=2
Exp 3, parameter: 2000rpm, 1.00M, 230°C, results: Coverage=0.963, Uniformity=3.7, Phase purity=1
Exp 4, parameter: 1500rpm, 0.40M, 170°C, results: Coverage=0.985, Uniformity=4.6, Phase purity=1
Exp 5, parameter: 1000rpm, 1.10M, 130°C, results: Coverage=0.989, Uniformity=4.3, Phase purity=1
Exp 6, parameter: 1500rpm, 1.10M, 110°C, results: Coverage=0.905, Uniformity=4.1, Phase purity=1
Exp 7, parameter: 2500rpm, 1.50M, 150°C, results: Coverage=0.926, Uniformity=4.0, Phase purity=1
Exp 8, parameter: 2000rpm, 0.30M, 190°C, results: Coverage=0.972, Uniformity=5.3, Phase purity=2
Exp 9, parameter: 1000rpm, 0.50M, 240°C, results: Coverage=0.960, Uniformity=7.6, Phase purity=1

In [17]:
# ----- Objective definitions -----
obj_1 = "coverage"        # maximize (0..1)
obj_2 = "uniformity"      # minimize (better uniformity = lower values)
obj_3 = "phase_purity"  # CONVERTED TO CONSTRAINT: must equal 1 (single phase)

# ----- Parameter ranges and discrete values -----
param_ranges = {
    "spin_speed_rpm": [500.0, 3000.0],
    "precursor_conc_mol_L": [0.1, 1.6],
    "anneal_temp_C": [100.0, 300.0]
}

spinspeed_vals = list(np.arange(500, 3000, 500, dtype=float))
conc_vals = [round(x, 2) for x in np.arange(0.1, 1.6, 0.1)]
anneal_vals = list(np.arange(100, 300, 10, dtype=float))
param_lists = [spinspeed_vals, conc_vals, anneal_vals]
param_names = ["spin_speed_rpm", "precursor_conc_mol_L", "anneal_temp_C"]

PARAMS = [
    {"name": "spin_speed_rpm_norm", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
    {"name": "precursor_conc_mol_L_norm", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
    {"name": "anneal_temp_C_norm", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
]

# ----- Initial experimental data (Latin hypercube sampling) -----
X_train = pd.DataFrame([
    {"spin_speed_rpm": 2000, "precursor_conc_mol_L": 0.6, "anneal_temp_C": 250.0},
    {"spin_speed_rpm": 500, "precursor_conc_mol_L": 1.3, "anneal_temp_C": 150.0},
    {"spin_speed_rpm": 2000, "precursor_conc_mol_L": 1.4, "anneal_temp_C": 270.0},
    {"spin_speed_rpm": 2000, "precursor_conc_mol_L": 1.0, "anneal_temp_C": 230.0},
    {"spin_speed_rpm": 1500, "precursor_conc_mol_L": 0.4, "anneal_temp_C": 170.0},
    {"spin_speed_rpm": 1000, "precursor_conc_mol_L": 1.1, "anneal_temp_C": 130.0},
    {"spin_speed_rpm": 1500, "precursor_conc_mol_L": 1.1, "anneal_temp_C": 110.0},
    {"spin_speed_rpm": 2500, "precursor_conc_mol_L": 1.5, "anneal_temp_C": 150.0},
    {"spin_speed_rpm": 2000, "precursor_conc_mol_L": 0.3, "anneal_temp_C": 190.0},
    {"spin_speed_rpm": 1000, "precursor_conc_mol_L": 0.5, "anneal_temp_C": 240.0},
    {"spin_speed_rpm": 2000, "precursor_conc_mol_L": 0.6, "anneal_temp_C": 220.0},
    {"spin_speed_rpm": 1500, "precursor_conc_mol_L": 0.9, "anneal_temp_C": 280.0}
])

# Normalize training data
X_train_array = X_train[param_names].values
X_train_normalized = x_normalizer(X_train_array, [list(param_ranges[p]) for p in param_names])
X_train_norm = pd.DataFrame(X_train_normalized, columns=[f"{p}_norm" for p in param_names])

# Experimental results (raw values)
y_train_raw = pd.DataFrame([
    {"coverage": 0.936, "uniformity": 7.108, "phase_purity": 2},
    {"coverage": 0.783, "uniformity": 5.452, "phase_purity": 2},
    {"coverage": 0.984, "uniformity": 4.965, "phase_purity": 2},
    {"coverage": 0.963, "uniformity": 3.682, "phase_purity": 1},
    {"coverage": 0.985, "uniformity": 4.581, "phase_purity": 1},
    {"coverage": 0.989, "uniformity": 4.331, "phase_purity": 1},
    {"coverage": 0.905, "uniformity": 4.141, "phase_purity": 1},
    {"coverage": 0.926, "uniformity": 4.028, "phase_purity": 1},
    {"coverage": 0.972, "uniformity": 5.35, "phase_purity": 2},
    {"coverage": 0.96, "uniformity": 7.561, "phase_purity": 1},
    {"coverage": 0.958, "uniformity": 4.462, "phase_purity": 3},
    {"coverage": 0.976, "uniformity": 7.347, "phase_purity": 2}
])

y_train_raw['phase_purity_violation'] = np.where(
    y_train_raw['phase_purity'] == 1, 
    -0.1,  # Feasible: negative value (satisfies constraint <= 0)
    0.9    # Infeasible: positive value (violates constraint <= 0)
)

# Prepare data for BO (two objectives + one constraint)
y_train_bo = [
    {obj_1: y_train_raw.iloc[i][obj_1], 
     obj_2: y_train_raw.iloc[i][obj_2],
     'phase_purity_violation': y_train_raw.iloc[i]['phase_purity_violation']} 
    for i in range(len(y_train_raw))
]

gs = GenerationStrategy(steps=[
    GenerationStep(
        model=Models.BOTORCH_MODULAR,    # Pure BoTorch MOO step (skip Sobol)
        num_trials=-1,                   # unlimited
        max_parallelism=32,              # High parallelism for batch generation
    ),
])

ax_client = AxClient(generation_strategy=gs, verbose_logging=False)
# Define the constraint: phase_purity_violation <= 0 (i.e., phase_purity == 1)
phase_purity_violation_metric = Metric(name="phase_purity_violation", lower_is_better=True)

# Constraint: must be <= 0 to be feasible
phase_purity_constraint = OutcomeConstraint(
    metric=phase_purity_violation_metric,
    op=ComparisonOp.LEQ,  
    bound=0.0,
    relative=False,
)

ax_client.create_experiment(
    parameters=PARAMS,
    objectives={
        obj_1: ObjectiveProperties(minimize=False),          # maximize coverage
        obj_2: ObjectiveProperties(minimize=True),          # minimize log(uniformity)
    },
    outcome_constraints=["phase_purity_violation <= 0.0"],
)

# Seed the experiment with scaled objective data
for i in range(len(X_train_norm)):
    ax_client.attach_trial(X_train_norm.iloc[i].to_dict())
    ax_client.complete_trial(trial_index=i, raw_data=y_train_bo[i])


[INFO 09-22 20:27:38] ax.service.utils.instantiation: Due to non-specification, we will use the heuristic for selecting objective thresholds.
[INFO 09-22 20:27:38] ax.service.utils.instantiation: Created search space: SearchSpace(parameters=[RangeParameter(name='spin_speed_rpm_norm', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='precursor_conc_mol_L_norm', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='anneal_temp_C_norm', parameter_type=FLOAT, range=[0.0, 1.0])], parameter_constraints=[]).
[INFO 09-22 20:27:38] ax.core.experiment: Attached custom parameterizations [{'spin_speed_rpm_norm': 0.6, 'precursor_conc_mol_L_norm': 0.333333, 'anneal_temp_C_norm': 0.75}] as trial 0.
[INFO 09-22 20:27:38] ax.core.experiment: Attached custom parameterizations [{'spin_speed_rpm_norm': 0.0, 'precursor_conc_mol_L_norm': 0.8, 'anneal_temp_C_norm': 0.25}] as trial 1.
[INFO 09-22 20:27:38] ax.core.experiment: Attached custom parameterizations [{'spin_speed_rpm_norm': 0.6, 

In [18]:
def generate_next_experiments(n_suggest=12):
    """Generate the next batch of experiments using improved BO"""
    
    global ax_client  # Declare global at the beginning
    
    # Generate suggestions
    print(f"Generating {n_suggest} new experimental suggestions...")
    
    suggestions_normalized = []
    for i in range(n_suggest):
        try:
            params_norm, trial_index = ax_client.get_next_trial()
            arm_name = ax_client.experiment.trials[trial_index].arm.name
            suggestions_normalized.append({
                "trial_index": trial_index, 
                "arm_name": arm_name, 
                **params_norm
            })
            print(f"  Generated suggestion {i+1}/{n_suggest}")
        except Exception as e:
            print(f" Error generating suggestion {i+1}: {e}")
            break

    if not suggestions_normalized:
        print("Failed to generate any suggestions!")
        return None, None

    print(f" Successfully generated {len(suggestions_normalized)} suggestions!")
    if len(suggestions_normalized) >= n_suggest:
        print(f"SUCCESS: Generated requested {n_suggest} suggestions using custom BoTorch strategy!")
    
    suggestions_df = pd.DataFrame(suggestions_normalized)

    # Convert to original parameter space
    norm_cols = [f"{p}_norm" for p in param_names]
    X_suggestions_norm = suggestions_df[norm_cols].values
    X_suggestions_denorm = x_denormalizer(X_suggestions_norm, [list(param_ranges[p]) for p in param_names])
    
    suggestions_original = suggestions_df.copy()
    for i, param in enumerate(param_names):
        suggestions_original[param] = [x[i] for x in X_suggestions_denorm]

    # Snap to discrete values for practical experiments
    X_suggestions_discrete = get_closest_array(X_suggestions_denorm, param_lists)
    suggestions_discrete = pd.DataFrame(X_suggestions_discrete, columns=param_names)
    
    return suggestions_discrete, suggestions_original


next_experiments, next_experiments_continuous = generate_next_experiments(n_suggest=10)
print(next_experiments, next_experiments_continuous)
next_experiments.to_csv("/Users/shengfang/Desktop/TRI/test_FAPbI3/LLM_BO_10_round2.csv", index=False)
    

Generating 10 new experimental suggestions...
  Generated suggestion 1/10


/Users/shengfang/anaconda3/envs/BO/lib/python3.12/site-packages/ax/core/data.py:288: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


  Generated suggestion 2/10


/Users/shengfang/anaconda3/envs/BO/lib/python3.12/site-packages/ax/core/data.py:288: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


  Generated suggestion 3/10


...

  Generated suggestion 8/10


/Users/shengfang/anaconda3/envs/BO/lib/python3.12/site-packages/ax/core/data.py:288: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


  Generated suggestion 9/10


/Users/shengfang/anaconda3/envs/BO/lib/python3.12/site-packages/ax/core/data.py:288: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return cls(df=pd.concat(dfs, axis=0, sort=True))


  Generated suggestion 10/10
 Successfully generated 10 suggestions!
SUCCESS: Generated requested 10 suggestions using custom BoTorch strategy!
   spin_speed_rpm  precursor_conc_mol_L  anneal_temp_C
0          1000.0                   1.0          200.0
1          2000.0                   1.0          210.0
2          2000.0                   1.0          210.0
3          2500.0                   1.0          220.0
4          1000.0                   0.5          130.0
5          2000.0                   1.1          130.0
6          1000.0                   0.4          120.0
7          2500.0                   1.0          200.0
8          1000.0                   1.1          200.0
9          2000.0                   0.5          130.0    trial_index arm_name  spin_speed_rpm_norm  precursor_conc_mol_L_norm  \
0           12     12_0             0.263534                   0.607557   
1           13     13_0             0.518111                   0.600145   
2           14     14_0   

In [23]:
ax_client.save_to_json_file("/Users/shengfang/Desktop/TRI/test_FAPbI3/LLM_BO_Round1_model.json")  # This is the important one

In [ ]:
import json
def load_synthesis_knowledge(json_path):
    """Load synthesis knowledge from JSON file"""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            knowledge = json.load(f)
        print(f" Loaded synthesis knowledge for {knowledge.get('target_composition', 'Unknown')}")
        return knowledge
    except FileNotFoundError:
        print(f" File not found: {json_path}")
        return None
    except json.JSONDecodeError:
        print(f" Invalid JSON format in: {json_path}")
        return None

def extract_parameter_insights(knowledge, method="filtered"):
    """Extract synthesis insights from knowledge base
    
    Args:
        knowledge: Loaded JSON knowledge base
        method: "filtered" (clean format) or "raw" (full JSON as string)
    """
    
    if method == "raw":
        # Option 1: Send entire JSON as string
        return json.dumps(knowledge, indent=2)
    
    elif method == "filtered":
        # Option 2: Extract and format only synthesis content (current approach)
        insights = []
        
        if 'sections' in knowledge:
            for section_name, section_data in knowledge['sections'].items():
                content = section_data.get('content', '')
                if content and content != 'No content':
                    insights.append(f"## {section_name}\n{content}\n")
        
        return "\n".join(insights)
    
    else:
        raise ValueError("Method must be 'filtered' or 'raw'")

sample_conditions = []
for i, (index, row) in enumerate(next_experiments.iterrows()):
    condition = {
        'sample_id': f"Sample_{i+1}",
        'spin_speed': row[next_experiments.columns[0]],  # Spin Speed [rpm]
        'concentration': row[next_experiments.columns[1]],  # Precursor Concentration [mol/L]
        'annealing_temp': row[next_experiments.columns[2]]  # Annealing Temperature [degC]
    }
    sample_conditions.append(condition)

def llm_filter_samples(synthesis_knowledge, sample_conditions, target_count=6):
    """Use LLM to filter samples based on synthesis knowledge confidence"""
    
    # Format samples for LLM
    samples_text = ""
    for i, condition in enumerate(sample_conditions, 1):
        samples_text += f"""
Sample {i}:
- Spin Speed: {condition['spin_speed']:.0f} rpm
- Concentration: {condition['concentration']:.3f} mol/L
- Annealing Temperature: {condition['annealing_temp']:.0f}°C
"""

    # Get synthesis insights
    insights = extract_parameter_insights(synthesis_knowledge, method="filtered")
    
    prompt = f"""
Based on the experimental results and synthesis knowledge, evaluate and rank the experimental conditions below. Give more importance to conditions that align with good experimental results.

EXPERIMENTAL RESULTS SUMMARY:
{data_context}

SYNTHESIS KNOWLEDGE:
{insights}

EXPERIMENTAL CONDITIONS TO EVALUATE:
{samples_text}

Please select the TOP {target_count} experimental conditions that are most likely to produce high-quality alpha-phase FAPbI3 films based on the literature evidence.

For each selected condition, provide:
1. **Sample ID** (e.g., Sample 1, Sample 2, etc.)
2. **Confidence Score** (0-1, where 1 = highest confidence)
3. **Rationale** explaining why this condition is promising based on the experimental results and synthesis knowledge.

Rank them from highest to lowest confidence and provide ONLY the top {target_count} conditions.

Format your response clearly listing each selected sample with its confidence score and rationale.
"""

    try:
        print(f"Querying LLM to select top {target_count} experimental conditions...")
        response = ollama.chat(
            model='llama3.2',
            messages=[{
                'role': 'user',
                'content': prompt
            }],
            options={
                'temperature': 0.1,
                'top_p': 1.0,
                'num_predict': 2048
            }
        )
        
        return response['message']['content']
        
    except Exception as e:
        print(f"Error querying LLM for sample filtering: {e}")
        return None

# Load the synthesis knowledge JSON file
knowledge_path = "/Users/shengfang/Desktop/TRI/test_FAPbI3/fapbi3_synthesis_knowledge.json"
synthesis_knowledge = load_synthesis_knowledge(knowledge_path)

if synthesis_knowledge:
    llm_filtering_result = llm_filter_samples(synthesis_knowledge, sample_conditions, target_count=6)
    
    if llm_filtering_result:
        print("LLM SAMPLE SELECTION RESULTS:")
        print("="*100)
        print(llm_filtering_result)
        print("="*100)
        
        # Parse LLM response to extract selected sample IDs (only numbered selections, not mentions)
        import re
        # Look for patterns like "1. **Sample X**" or "1. Sample X" (actual selections)
        sample_pattern = r'^\d+\.\s+\*\*Sample\s+(\d+)\*\*|^\d+\.\s+Sample\s+(\d+)'
        selected_sample_ids = []
        
        for line in llm_filtering_result.split('\n'):
            match = re.search(sample_pattern, line.strip())
            if match:
                # Get the captured group that's not None
                sample_num = int(match.group(1) if match.group(1) else match.group(2))
                if sample_num <= len(next_experiments):  # Ensure sample ID is valid
                    selected_sample_ids.append(sample_num - 1)  # Convert to 0-based index
        
        # Remove duplicates and sort
        selected_sample_ids = sorted(list(set(selected_sample_ids)))
        
        
        if selected_sample_ids:
            # Create filtered DataFrame with LLM-selected samples
            filter_df = next_experiments.iloc[selected_sample_ids].copy()
            filter_df.reset_index(drop=True, inplace=True)  # Reset index for clean display
            
            print(f"\n FILTERED SAMPLES (LLM SELECTED - {len(filter_df)} samples):")
            print("="*100)
            
            
            # Format for clean display
            print(filter_df.round(3).to_string(index=False, float_format='%.3f'))
            print("="*100)
            
            # Save filtered samples
            filter_df.to_csv("/Users/shengfang/Desktop/TRI/test_FAPbI3/LLM_BO_10_round2_filtered.csv", index=True)


        else:
            print("\n No valid sample IDs extracted from LLM response - using first 6 samples as fallback")
            filter_df = next_experiments.head(6).copy()
            filter_df.reset_index(drop=True, inplace=True)

        # Display filtered samples
        print(f"\n FILTERED SAMPLES (LLM SELECTED - {len(filter_df)} samples):")
        print("="*100)

        # Format for clean display
        print(filter_df.round(3).to_string(index=False, float_format='%.3f'))
        print("="*100)

        # Save filtered samples
        filter_df.to_csv("/Users/shengfang/Desktop/TRI/test_FAPbI3/LLM_BO_10_round2._filtered.csv", index=True)

    else:
        print("Failed to get LLM filtering results")
else:
    print("No synthesis knowledge available for filtering")
    print("="*100)

 Loaded synthesis knowledge for FAPbI3
Querying LLM to select top 6 experimental conditions...
LLM SAMPLE SELECTION RESULTS:
Based on the literature evidence, I recommend the following top 6 experimental conditions for producing high-quality alpha-phase FAPbI3 films:

1. **Sample 2**
   - Spin Speed: 2000 rpm
   - Concentration: 1.000 mol/L
   - Annealing Temperature: 210°C
   - Confidence Score: 1
   - Rationale: This condition is promising because it closely matches the optimal processing parameters mentioned in the literature, including a high spin speed and annealing temperature.

2. **Sample 6**
   - Spin Speed: 2000 rpm
   - Concentration: 1.100 mol/L
   - Annealing Temperature: 130°C
   - Confidence Score: 1
   - Rationale: This condition is promising because it uses a high concentration of FAI, which is known to improve film quality, and an annealing temperature that is lower than the optimal value mentioned in the literature.

3. **Sample 5**
   - Spin Speed: 1000 rpm
   - Con